# Collect local experience

Run a typed XDRL interaction directly as a TorchRL policy.

In [ ]:
import torch
from torch import nn
from tensordict.nn import TensorDictModule
from torchrl.collectors import Collector
from torchrl.envs import PendulumEnv
from xdrl import (
    BatchSemantics,
    InteractionContract,
    InteractionPhase,
    KeyPresence,
    KeyRole,
    KeySchema,
    ModelRole,
    RuntimeInteractionContext,
    TensorDictSchema,
)


class PendulumPolicy(nn.Module):
    def forward(self, angle, velocity):
        return torch.tanh((angle + 0.1 * velocity).unsqueeze(-1))


env = PendulumEnv()
policy = TensorDictModule(
    PendulumPolicy(),
    in_keys=["th", "thdot"],
    out_keys=["action"],
)

In [ ]:
batch_dims = BatchSemantics(())
contract = InteractionContract(
    identity="pendulum:collection",
    role=ModelRole.ACTOR,
    phase=InteractionPhase.COLLECTION,
    module_path="policy",
    input_schema=TensorDictSchema(
        (
            KeySchema("th", KeyRole.OBSERVATION, KeyPresence.REQUIRED),
            KeySchema("thdot", KeyRole.OBSERVATION, KeyPresence.REQUIRED),
        ),
        batch_dims,
    ),
    output_schema=TensorDictSchema(
        (KeySchema("action", KeyRole.ACTION, KeyPresence.PRODUCED),),
        batch_dims,
    ),
    module_training=False,
)
collection = RuntimeInteractionContext(contract, policy, env.reset())

In [ ]:
collector = Collector(
    env,
    policy=collection,
    frames_per_batch=16,
    total_frames=32,
)
try:
    rollouts = list(collector)
finally:
    collector.shutdown()

assert len(rollouts) == 2
assert rollouts[0]["action"].shape == (16, 1)
{
    "batches": len(rollouts),
    "frames_per_batch": rollouts[0].numel(),
    "action_shape": tuple(rollouts[0]["action"].shape),
    "model_events": len(collection.events),
}